In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/rewritten/o4_mini_high_small/checkpoints/post_cell_9_try_0.pickle

In [ ]:
%%cudf.pandas.profile
### cell 10 ###

# GPU-optimized version
# 1) pull down only the 3 columns of interest and drop all‐null rows in one GPU call
movies_with_scores = m[["imdb_score", "gross", "budget"]].dropna()

# 2) alias each series once to avoid repeated CPU-side __getitem__
imdb   = movies_with_scores.imdb_score
ngross  = movies_with_scores.gross
nbudget = movies_with_scores.budget

# 3) compute false-positive rate in a single pipeline:
#    (gross < budget) masked by imdb>6, then take the mean of the resulting boolean mask
false_positive_rate = ( (ngross < nbudget).where(imdb > 6) ).mean()

false_positive_rate